# 06 — Analyse-Vorbereitung (Teil 1): Aufbau der 20 MI-Datensätze

Dieses Notebook baut aus den abgeschlossenen Multiple-Imputation-Artefakten (m = 20) die **20 vollständigen Analysedatensätze** für die spätere, MI-korrekte Analyse (REWB, BMA, Elastic Net). Es entspricht dem Skript `code/06_analysis_prep.R`.

**Wichtig:** Hier finden **keine** Transformationen, **keine** Standardisierung und **keine** Modelle statt — das ist Teil 2. Panel: 119 Länder × 2007–2022 = **1.904 Zeilen**.

Jede Version *j* = ISO3, Country_Name, Year + HDI + Happiness_Score + 35 Features + 4 `META_`-Merkmale (**44 Spalten**). Die Metadaten (F1–F4) und HDI sind über alle 20 Versionen identisch; nur die imputierten Zellen streuen (Abbild der Imputations-Unsicherheit).

**Output:** `code/Created Files for Analysis/analysis_base.rds` — benannte Liste `imp01`…`imp20`.

## 0 — Setup: Pakete, Pfade & Repo-Root

Lädt `mice` (für `complete()`), bestimmt robust den Repo-Root (funktioniert, egal ob aus `code/` oder dem Projektwurzel-Verzeichnis gestartet) und legt die Ein-/Ausgabepfade fest. Das Skript ist **deterministisch** (keine Zufallszahlen).

In [1]:
suppressMessages(library(mice))

root <- getwd()
if (basename(root) == "code") root <- dirname(root)
stopifnot(dir.exists(file.path(root, "Created DBs", "Cleaned DBs")),
          dir.exists(file.path(root, "code", "Created Files for Analysis")))

p_step1 <- file.path(root, "code", "Created Files for Analysis", "step1_lifesat_pmm.rds")
p_step2 <- file.path(root, "code", "Created Files for Analysis", "step2_features_amelia.rds")
p_targ  <- file.path(root, "Created DBs", "Cleaned DBs", "db1_targets_cleaned.csv")
p_clean <- file.path(root, "Created DBs", "Cleaned DBs", "cleaned_database.csv")
p_out   <- file.path(root, "code", "Created Files for Analysis", "analysis_base.rds")

cat("== 06_analysis_prep.R : Aufbau der 20 MI-Analysedatensaetze ==\n")
cat("Repo-Root:", root, "\n\n")

== 06_analysis_prep.R : Aufbau der 20 MI-Analysedatensaetze ==


Repo-Root: C:/Users/maier/OneDrive/Dokumente/_Studium/Bachelorarbeit/Practical Analysis/Explaining-Well-Being 



## 1 — Inputs laden & Spaltenstruktur

Vier Eingaben: die beiden Imputations-Artefakte (`step1` = mice-PMM für *Happiness_Score*, `step2` = Amelia für die 35 Features), `db1_targets_cleaned.csv` (HDI, nie imputiert) und `cleaned_database.csv` als **Referenz-Spaltenstruktur** und Quelle der 4 `META_`-Merkmale (F1–F4). Aus der Referenz werden die 44 Zielspalten und die 35 Feature-Namen abgeleitet.

In [2]:
step1 <- readRDS(p_step1)                         # mice::mids, m=20 (Happiness_Score)
step2 <- readRDS(p_step2)                         # Amelia,     m=20 (35 Features)
targ  <- read.csv(p_targ,  check.names = FALSE, stringsAsFactors = FALSE)
cd    <- read.csv(p_clean, check.names = FALSE, stringsAsFactors = FALSE)

M <- 20L
stopifnot(step1$m == M, step2$m == M)

## Referenz-Spaltenstruktur (44 Spalten) aus cleaned_database
cols_all  <- names(cd)                            # Zielreihenfolge des Outputs
meta_cols <- c("META_WEO_Gruppe", "META_Lambda_BIP",
               "META_LB_vs_Global", "META_LB_vs_Gruppe")   # F1..F4
key_cols  <- c("ISO3", "Country_Name", "Year")
tgt_cols  <- c("HDI", "Happiness_Score")
feat_orig <- setdiff(cols_all, c(key_cols, tgt_cols, meta_cols))   # 35 Features
stopifnot(length(feat_orig) == 35L)

## 2 — Namens-Rückmapping (Amelia → Original)

Amelia hat die Feature-Namen syntaktisch sanitisiert (`gsub("[^A-Za-z0-9]", "_", …)`). Hier wird die Rückabbildung auf die Originalnamen (z. B. `WB-WDI-NY-GDP-PCAP-KD`) gebildet und geprüft, dass alle 35 eindeutig zugeordnet sind.

In [3]:
feat_san <- gsub("[^A-Za-z0-9]", "_", feat_orig)
am_names <- names(step2$imputations[[1]])
stopifnot(all(feat_san %in% am_names),            # alle 35 gefunden
          !any(duplicated(feat_san)))             # eindeutig / bijektiv
cat("Namens-Rueckmapping: alle 35 Features eindeutig zugeordnet.\n")

Namens-Rueckmapping: alle 35 Features eindeutig zugeordnet.


## 3 — META-Referenztabelle (ISO3 + Year → F1–F4)

Die vier Klassifikationsmerkmale werden je **ISO3 + Year** bereitgestellt — nicht nur je ISO3, weil `META_WEO_Gruppe` (F1) und `META_LB_vs_Gruppe` (F4) über die Jahre variieren können. F3/F4 sind in der CSV als **Leerstring** kodiert, wo λ fehlt → hier zu `NA` normalisiert. Die λ-basierten Merkmale fehlen für genau 6 Länder (AFG, ARE, NIC, PSE, ROU, SLV).

In [4]:
## F3/F4 sind Character mit "" fuer die 6 Laender -> "" als NA behandeln.
meta_ref <- cd[, c("ISO3", "Year", meta_cols)]
for (m in c("META_LB_vs_Global", "META_LB_vs_Gruppe")) {
  meta_ref[[m]][meta_ref[[m]] == ""] <- NA
}
key_meta <- paste(meta_ref$ISO3, meta_ref$Year)

## Ziel-/HDI-Lookups
key_targ <- paste(targ$ISO3, targ$Year)

## Laender ohne Lambda (erwartete Restluecken bei F2/F3/F4)
lambda_missing_iso <- c("AFG", "ARE", "NIC", "PSE", "ROU", "SLV")

## 4 — Aufbau der 20 Versionen

Für jede Imputation *j* = 1…20 wird ein vollständiger Datensatz zusammengesetzt: Schlüssel + 35 Features (aus Amelia, Originalnamen) + `Happiness_Score` (aus `mice::complete`, volle Präzision) + `HDI` (aus db1_targets) + F1–F4 (Join per ISO3 + Year). Spaltenreihenfolge exakt wie `cleaned_database` (44 Spalten), Zeilen nach ISO3, Year sortiert.

In [5]:
build_one <- function(j) {
  am <- step2$imputations[[j]]                    # ISO3, Country_Name, Year + 35 (sanitisiert)

  df <- am[, key_cols]
  feat <- am[, feat_san, drop = FALSE]            # 35 Features
  names(feat) <- feat_orig                        # Originalnamen
  df <- cbind(df, feat)

  key_df <- paste(df$ISO3, df$Year)

  ## Happiness_Score aus der j-ten mice-Vervollstaendigung (volle Praezision)
  cj <- complete(step1, j)
  df$Happiness_Score <- cj$Happiness_Score[match(key_df, paste(cj$ISO3, cj$Year))]

  ## HDI (nie imputiert) aus db1_targets_cleaned
  df$HDI <- targ$HDI[match(key_df, key_targ)]

  ## F1..F4 per ISO3+Year (in allen 20 Versionen identisch)
  mi <- match(key_df, key_meta)
  for (m in meta_cols) df[[m]] <- meta_ref[[m]][mi]

  ## Spaltenreihenfolge exakt wie cleaned_database (44 Spalten), Zeilen ISO3+Year
  df <- df[, cols_all]
  df <- df[order(df$ISO3, df$Year), ]
  rownames(df) <- NULL
  df
}

cat("\nBaue 20 Versionen ...\n")
imp_list <- lapply(seq_len(M), build_one)
names(imp_list) <- sprintf("imp%02d", seq_len(M))

## Referenz (cleaned) in gleiche Zeilenreihenfolge bringen
cd_sorted <- cd[order(cd$ISO3, cd$Year), ]
rownames(cd_sorted) <- NULL
ref_key   <- paste(cd_sorted$ISO3, cd_sorted$Year)


Baue 20 Versionen ...


## 5 — Pflicht-Asserts

Sieben Prüfungen (A1–A7). Bei jeder Verletzung erhöht sich `n_fail`; der Output wird dann in Schritt 7 **nicht** geschrieben. Zusätzlich ein rein informativer Check der Happiness-Pool-Abweichung (die `cleaned_database` rundet Happiness auf ~3 Dezimalstellen; die 20 DBs tragen die ungerundeten Rohwerte).

In [6]:
n_fail <- 0L
check <- function(name, ok, detail = "") {
  status <- if (isTRUE(ok)) "PASS" else "FAIL"
  if (!isTRUE(ok)) n_fail <<- n_fail + 1L
  cat(sprintf("  [%s] %s%s\n", status, name,
              if (nzchar(detail)) paste0("  ->  ", detail) else ""))
}

cat("\n== Pflicht-Asserts ==\n")

## A1: 1904 Zeilen (119 x 16), identische Zeilenreihenfolge ueber alle 20
dims_ok <- all(vapply(imp_list, nrow, 0L) == 1904L)
n_ctry  <- length(unique(imp_list[[1]]$ISO3)); n_year <- length(unique(imp_list[[1]]$Year))
order_ok <- all(vapply(imp_list, function(d) identical(paste(d$ISO3, d$Year), ref_key), NA))
check("A1 Zeilen=1904 & Reihenfolge identisch", dims_ok && order_ok && n_ctry == 119L && n_year == 16L,
      sprintf("%d Laender x %d Jahre; alle 20 gleich sortiert=%s", n_ctry, n_year, order_ok))

## A2: 0 NA ausser F2/F3/F4 (je 96 NA, genau bei den 6 Laendern); F1 vollstaendig
nonmeta <- setdiff(cols_all, meta_cols)
na_nonmeta <- vapply(imp_list, function(d) sum(is.na(d[, nonmeta])), 0L)
na_f1 <- vapply(imp_list, function(d) sum(is.na(d$META_WEO_Gruppe)), 0L)
lam_cols <- c("META_Lambda_BIP", "META_LB_vs_Global", "META_LB_vs_Gruppe")
na_lam_ok <- all(vapply(imp_list, function(d) {
  all(vapply(lam_cols, function(m) {
    idx <- is.na(d[[m]])
    sum(idx) == 96L && setequal(unique(d$ISO3[idx]), lambda_missing_iso)
  }, NA))
}, NA))
check("A2 NA-Bilanz (nur F2/F3/F4 je 96 NA an 6 Laendern)",
      all(na_nonmeta == 0L) && all(na_f1 == 0L) && na_lam_ok,
      sprintf("Non-META NA=%d, F1 NA=%d, F2/F3/F4-Muster ok=%s",
              max(na_nonmeta), max(na_f1), na_lam_ok))

## A3: Spaltenset identisch mit cleaned_database (44 Spalten)
set_ok <- all(vapply(imp_list, function(d) setequal(names(d), cols_all) && ncol(d) == length(cols_all), NA))
check("A3 Spaltenset == cleaned_database (44 Spalten)", set_ok,
      sprintf("%d Spalten je Version", ncol(imp_list[[1]])))

## A4: Mittelwert jedes Features ueber 20 Versionen ~ cleaned_database (rel. Tol 1e-6)
feat_maxrel <- 0; feat_worst <- ""
for (f in feat_orig) {
  Mv <- vapply(imp_list, function(d) d[[f]], numeric(1904))
  mu <- rowMeans(Mv)
  ref <- cd_sorted[[f]]
  rel <- max(abs(mu - ref) / pmax(abs(ref), 1e-8))
  if (rel > feat_maxrel) { feat_maxrel <- rel; feat_worst <- f }
}
check("A4 Feature-Pool-Mittel ~ cleaned (rel<1e-6)", feat_maxrel < 1e-6,
      sprintf("max rel. Abw. = %.2e (%s)", feat_maxrel, feat_worst))

## A5: Beobachtete (nicht imputierte) Zellen ueber alle 20 Versionen identisch
##     Features: Beobachtung via Amelia-missMatrix; Happiness via mice-where.
mm <- step2$missMatrix
colnames(mm) <- ifelse(colnames(mm) %in% feat_san,
                       feat_orig[match(colnames(mm), feat_san)], colnames(mm))
## Reihenfolge der missMatrix = Original-Amelia-Reihenfolge (ISO3,Year wie step2$imputations[[1]])
amel_key <- paste(step2$imputations[[1]]$ISO3, step2$imputations[[1]]$Year)
obs_identical <- TRUE; obs_detail <- ""
for (f in feat_orig) {
  obs_rows_amord <- !mm[, f]
  Mv <- vapply(imp_list, function(d) d[[f]], numeric(1904))       # sortiert nach ref_key
  ## missMatrix-Zeilen auf ref_key-Reihenfolge umsortieren
  obs_rows <- obs_rows_amord[match(ref_key, amel_key)]
  if (any(obs_rows)) {
    rng <- apply(Mv[obs_rows, , drop = FALSE], 1, function(x) max(x) - min(x))
    if (max(rng) > 0) { obs_identical <- FALSE; obs_detail <- paste0("Feature ", f) }
  }
}
## Happiness: beobachtete Zellen identisch ueber die 20 Versionen
Hs <- vapply(imp_list, function(d) d$Happiness_Score, numeric(1904))
hap_obs_amord <- !step1$where[, "Happiness_Score"]
hap_obs <- hap_obs_amord[match(ref_key, paste(step1$data$ISO3, step1$data$Year))]
hap_rng <- apply(Hs[hap_obs, , drop = FALSE], 1, function(x) max(x) - min(x))
if (max(hap_rng) > 0) { obs_identical <- FALSE; obs_detail <- paste(obs_detail, "Happiness_Score") }
check("A5 beobachtete Zellen identisch ueber 20 Versionen", obs_identical,
      if (nzchar(obs_detail)) obs_detail else "Features + Happiness_Score ok")

## A6: F2, F3 je Land konstant; F1, F4 je Land x Jahr ueber alle 20 Versionen identisch
##     (Jahresvariation von F1/F4 ist erlaubt.)
const_ok <- TRUE
for (m in c("META_Lambda_BIP", "META_LB_vs_Global")) {
  v <- tapply(imp_list[[1]][[m]], imp_list[[1]]$ISO3,
              function(z) length(unique(z[!is.na(z)])) > 1)
  if (any(v, na.rm = TRUE)) const_ok <- FALSE
}
## F1..F4 ueber die 20 Versionen identisch (gleiche Metadaten-Quelle)
meta_same <- all(vapply(meta_cols, function(m) {
  base <- imp_list[[1]][[m]]
  all(vapply(imp_list, function(d) identical(d[[m]], base), NA))
}, NA))
check("A6 F2/F3 landkonstant & F1..F4 in allen 20 identisch", const_ok && meta_same,
      sprintf("F2/F3 konstant=%s, META ueber Versionen identisch=%s", const_ok, meta_same))

## A7: HDI in allen 20 Versionen identisch und == db1_targets
hdi_base <- imp_list[[1]]$HDI
hdi_same <- all(vapply(imp_list, function(d) identical(d$HDI, hdi_base), NA))
hdi_ref  <- targ$HDI[match(ref_key, key_targ)]
hdi_eq   <- max(abs(hdi_base - hdi_ref)) == 0
check("A7 HDI identisch ueber 20 & == db1_targets", hdi_same && hdi_eq,
      sprintf("ueber Versionen identisch=%s, max|Diff zu targets|=%g", hdi_same, max(abs(hdi_base - hdi_ref))))

## ---- Informativ: Happiness-Pool-Abweichung (Rundung in cleaned) ----
hap_mu  <- rowMeans(Hs)
hap_rel <- max(abs(hap_mu - cd_sorted$Happiness_Score) / pmax(abs(cd_sorted$Happiness_Score), 1e-8))
cat(sprintf("\n  [INFO] Happiness_Score Pool-Mittel vs. cleaned: max rel. Abw. = %.2e\n", hap_rel))
cat("         (erwartet ~2e-4; cleaned_database rundet Happiness, Rohwerte bleiben ungerundet)\n")
if (hap_rel > 1e-2) { n_fail <- n_fail + 1L; cat("  [FAIL] Happiness-Pool-Sanity (>1e-2)\n") }


== Pflicht-Asserts ==


  [PASS] A1 Zeilen=1904 & Reihenfolge identisch  ->  119 Laender x 16 Jahre; alle 20 gleich sortiert=TRUE


  [PASS] A2 NA-Bilanz (nur F2/F3/F4 je 96 NA an 6 Laendern)  ->  Non-META NA=0, F1 NA=0, F2/F3/F4-Muster ok=TRUE


  [PASS] A3 Spaltenset == cleaned_database (44 Spalten)  ->  44 Spalten je Version


  [PASS] A4 Feature-Pool-Mittel ~ cleaned (rel<1e-6)  ->  max rel. Abw. = 8.83e-14 (WB-WDI-NY-ADJ-DRES-GN-ZS)


  [PASS] A5 beobachtete Zellen identisch ueber 20 Versionen  ->  Features + Happiness_Score ok


  [PASS] A6 F2/F3 landkonstant & F1..F4 in allen 20 identisch  ->  F2/F3 konstant=TRUE, META ueber Versionen identisch=TRUE


  [PASS] A7 HDI identisch ueber 20 & == db1_targets  ->  ueber Versionen identisch=TRUE, max|Diff zu targets|=0



  [INFO] Happiness_Score Pool-Mittel vs. cleaned: max rel. Abw. = 2.12e-04


         (erwartet ~2e-4; cleaned_database rundet Happiness, Rohwerte bleiben ungerundet)


## 6 — Konsolen-Protokoll: Dimensionen & NA-Bilanz

Kompakte Übersicht je Version: Zeilen, Spalten, Gesamt-NA und META-NA (erwartet 288 = 3 × 96).

In [7]:
cat("\n== Dimensionen & NA-Bilanz je Version ==\n")
cat(sprintf("  %-6s %6s %5s %8s %8s\n", "Vers.", "Zeilen", "Spal.", "NA-ges", "NA-META"))
for (nm in names(imp_list)) {
  d <- imp_list[[nm]]
  cat(sprintf("  %-6s %6d %5d %8d %8d\n", nm, nrow(d), ncol(d),
              sum(is.na(d)), sum(is.na(d[, meta_cols]))))
}


== Dimensionen & NA-Bilanz je Version ==


  Vers.  Zeilen Spal.   NA-ges  NA-META


  imp01    1904    44      288      288
  imp02    1904    44      288      288
  imp03    1904    44      288      288
  imp04    1904    44      288      288
  imp05    1904    44      288      288
  imp06    1904    44      288      288
  imp07    1904    44      288      288
  imp08    1904    44      288      288
  imp09    1904    44      288      288
  imp10    1904    44      288      288
  imp11    1904    44      288      288
  imp12    1904    44      288      288
  imp13    1904    44      288      288
  imp14    1904    44      288      288
  imp15    1904    44      288      288
  imp16    1904    44      288      288
  imp17    1904    44      288      288
  imp18    1904    44      288      288
  imp19    1904    44      288      288
  imp20    1904    44      288      288


## 7 — Output speichern

Schreibt `analysis_base.rds` (benannte Liste `imp01`…`imp20`) **nur**, wenn alle Asserts bestanden haben.

In [8]:
if (n_fail == 0L) {
  saveRDS(imp_list, p_out)
  cat(sprintf("\nAlle Asserts PASS. Output geschrieben:\n  %s\n", p_out))
} else {
  stop(sprintf("%d Assert(s) fehlgeschlagen - Output NICHT geschrieben.", n_fail))
}


Alle Asserts PASS. Output geschrieben:
  C:/Users/maier/OneDrive/Dokumente/_Studium/Bachelorarbeit/Practical Analysis/Explaining-Well-Being/code/Created Files for Analysis/analysis_base.rds
